# 🛠️ FEATURE ENGINEERING - Predicción de Deserción Estudiantil
## Día 3: Creación de Features + Pipeline de Preprocessing
---
**Objetivo:** Crear features derivadas inteligentes basadas en insights del EDA y construir pipeline robusto.

**Duración estimada:** 4 horas

**Autor:** Jhordan Cotrina  
**Fecha:** Octubre 2025

**Insights clave del EDA:**
- Top features: Rendimiento académico 1er/2do semestre
- Regla detección temprana: <4 aprobadas + <11 promedio
- Target: 50% Graduate, 32% Dropout, 18% Enrolled

## 📦 1. IMPORTACIONES Y CONFIGURACIÓN

In [1]:
# Librerías estándar
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import json

# Scikit-learn preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin

# Configuración
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
np.random.seed(42)

print("✅ Librerías importadas correctamente")

✅ Librerías importadas correctamente


## 📥 2. CARGA DE DATOS

In [2]:
# Cargar dataset
df = pd.read_csv('../data/raw/student_data.csv')

print(f"📊 Dataset cargado: {df.shape[0]} filas × {df.shape[1]} columnas")
print(f"\nPrimeras columnas:")
print(df.columns.tolist()[:10])

📊 Dataset cargado: 4424 filas × 37 columnas

Primeras columnas:
['Marital status', 'Application mode', 'Application order', 'Course', 'Daytime/evening attendance\t', 'Previous qualification', 'Previous qualification (grade)', 'Nacionality', "Mother's qualification", "Father's qualification"]


## 🎨 3. FEATURE ENGINEERING - PARTE 1: ACADÉMICAS

Basado en insights del EDA, crearemos features derivadas del rendimiento académico.

In [3]:
# Crear copia para trabajar
df_engineered = df.copy()

print("🔧 Creando features académicas derivadas...")
print("="*60)

# 1. PROMEDIOS DE CALIFICACIONES
df_engineered['avg_grade_1st_sem'] = df['Curricular units 1st sem (grade)']
df_engineered['avg_grade_2nd_sem'] = df['Curricular units 2nd sem (grade)']
df_engineered['avg_grade_overall'] = (df['Curricular units 1st sem (grade)'] + 
                                       df['Curricular units 2nd sem (grade)']) / 2

# 2. TASAS DE APROBACIÓN
# Evitar división por cero
df_engineered['approval_rate_1st_sem'] = np.where(
    df['Curricular units 1st sem (enrolled)'] > 0,
    df['Curricular units 1st sem (approved)'] / df['Curricular units 1st sem (enrolled)'],
    0
)

df_engineered['approval_rate_2nd_sem'] = np.where(
    df['Curricular units 2nd sem (enrolled)'] > 0,
    df['Curricular units 2nd sem (approved)'] / df['Curricular units 2nd sem (enrolled)'],
    0
)

df_engineered['approval_rate_overall'] = (
    df_engineered['approval_rate_1st_sem'] + df_engineered['approval_rate_2nd_sem']
) / 2

# 3. TOTAL DE UNIDADES
df_engineered['total_approved'] = (df['Curricular units 1st sem (approved)'] + 
                                    df['Curricular units 2nd sem (approved)'])
df_engineered['total_enrolled'] = (df['Curricular units 1st sem (enrolled)'] + 
                                    df['Curricular units 2nd sem (enrolled)'])

# 4. CAMBIO DE RENDIMIENTO (1er → 2do semestre)
df_engineered['grade_improvement'] = (df['Curricular units 2nd sem (grade)'] - 
                                       df['Curricular units 1st sem (grade)'])
df_engineered['approval_improvement'] = (df['Curricular units 2nd sem (approved)'] - 
                                          df['Curricular units 1st sem (approved)'])

# 5. RATIO DE EVALUACIONES VS APROBADAS
df_engineered['eval_to_approved_1st'] = np.where(
    df['Curricular units 1st sem (approved)'] > 0,
    df['Curricular units 1st sem (evaluations)'] / df['Curricular units 1st sem (approved)'],
    0
)

df_engineered['eval_to_approved_2nd'] = np.where(
    df['Curricular units 2nd sem (approved)'] > 0,
    df['Curricular units 2nd sem (evaluations)'] / df['Curricular units 2nd sem (approved)'],
    0
)

print("✅ Features académicas creadas:")
new_academic_features = [
    'avg_grade_overall', 'approval_rate_overall', 'total_approved', 
    'grade_improvement', 'approval_improvement'
]
for feat in new_academic_features:
    print(f"  • {feat}")

print(f"\nTotal features académicas nuevas: {len([c for c in df_engineered.columns if c not in df.columns])}")

🔧 Creando features académicas derivadas...
✅ Features académicas creadas:
  • avg_grade_overall
  • approval_rate_overall
  • total_approved
  • grade_improvement
  • approval_improvement

Total features académicas nuevas: 12


## 🚨 4. FEATURE ENGINEERING - PARTE 2: INDICADORES DE RIESGO

Basado en la regla de detección temprana del EDA.

In [4]:
print("🚨 Creando indicadores de riesgo...")
print("="*60)

# 1. RIESGO ACADÉMICO TEMPRANO (basado en insight del EDA)
df_engineered['at_risk_1st_sem'] = (
    (df['Curricular units 1st sem (approved)'] < 4) & 
    (df['Curricular units 1st sem (grade)'] < 11)
).astype(int)

# 2. RENDIMIENTO MUY BAJO
df_engineered['very_low_performance'] = (
    df_engineered['avg_grade_overall'] < 10
).astype(int)

# 3. APROBACIÓN CRÍTICA
df_engineered['critical_approval'] = (
    df_engineered['approval_rate_overall'] < 0.5
).astype(int)

# 4. DETERIORO DE RENDIMIENTO
df_engineered['performance_declining'] = (
    df_engineered['grade_improvement'] < -2
).astype(int)

# 5. RIESGO ECONÓMICO
df_engineered['economic_risk'] = (
    (df['Tuition fees up to date'] == 0) | (df['Debtor'] == 1)
).astype(int)

# 6. SCORE DE RIESGO COMBINADO
df_engineered['risk_score'] = (
    df_engineered['at_risk_1st_sem'] * 3 +  # Peso más alto
    df_engineered['very_low_performance'] * 2 +
    df_engineered['critical_approval'] * 2 +
    df_engineered['performance_declining'] * 1 +
    df_engineered['economic_risk'] * 1
)

print("✅ Indicadores de riesgo creados:")
risk_features = [
    'at_risk_1st_sem', 'very_low_performance', 'critical_approval',
    'performance_declining', 'economic_risk', 'risk_score'
]
for feat in risk_features:
    print(f"  • {feat}")

# Analizar distribución del risk_score por clase
print(f"\n📊 Distribución de Risk Score por Target:")
print(df_engineered.groupby('Target')['risk_score'].describe())

🚨 Creando indicadores de riesgo...
✅ Indicadores de riesgo creados:
  • at_risk_1st_sem
  • very_low_performance
  • critical_approval
  • performance_declining
  • economic_risk
  • risk_score

📊 Distribución de Risk Score por Target:
           count      mean       std  min  25%  50%  75%  max
Target                                                       
Dropout   1421.0  4.259676  3.218474  0.0  1.0  5.0  7.0  9.0
Enrolled   794.0  1.202771  2.234193  0.0  0.0  0.0  1.0  9.0
Graduate  2209.0  0.329108  1.316998  0.0  0.0  0.0  0.0  8.0


## 💰 5. FEATURE ENGINEERING - PARTE 3: ECONÓMICAS Y DEMOGRÁFICAS

In [5]:
print("💰 Creando features económicas y demográficas...")
print("="*60)

# 1. ÍNDICE SOCIOECONÓMICO COMBINADO
df_engineered['socioeconomic_index'] = (
    df['Scholarship holder'] * 2 +  # Beca es positivo
    df['Tuition fees up to date'] * 1 -
    df['Debtor'] * 2  # Deuda es negativo
)

# 2. EDAD RELATIVA
median_age = df['Age at enrollment'].median()
df_engineered['age_deviation'] = df['Age at enrollment'] - median_age
df_engineered['is_mature_student'] = (df['Age at enrollment'] > 25).astype(int)
df_engineered['is_very_young'] = (df['Age at enrollment'] < 18).astype(int)

# 3. NIVEL EDUCATIVO PADRES COMBINADO
df_engineered['parents_education_avg'] = (
    df["Mother's qualification"] + df["Father's qualification"]
) / 2

# 4. CONTEXTO ECONÓMICO (si existen estas columnas)
if 'Unemployment rate' in df.columns and 'Inflation rate' in df.columns:
    df_engineered['economic_stress'] = (
        df['Unemployment rate'] + df['Inflation rate']
    ) / 2

print("✅ Features económicas/demográficas creadas:")
econ_features = [
    'socioeconomic_index', 'age_deviation', 'is_mature_student',
    'parents_education_avg'
]
for feat in econ_features:
    if feat in df_engineered.columns:
        print(f"  • {feat}")

💰 Creando features económicas y demográficas...
✅ Features económicas/demográficas creadas:
  • socioeconomic_index
  • age_deviation
  • is_mature_student
  • parents_education_avg


## 📊 6. ANÁLISIS DE NUEVAS FEATURES

In [6]:
# Comparar features originales vs nuevas
original_features = len(df.columns)
new_features = len(df_engineered.columns) - original_features

print("="*60)
print("RESUMEN DE FEATURE ENGINEERING")
print("="*60)
print(f"Features originales: {original_features}")
print(f"Features nuevas creadas: {new_features}")
print(f"Total features: {len(df_engineered.columns)}")

print(f"\n📋 Lista de nuevas features:")
new_cols = [col for col in df_engineered.columns if col not in df.columns]
for i, col in enumerate(new_cols, 1):
    print(f"  {i:2d}. {col}")

RESUMEN DE FEATURE ENGINEERING
Features originales: 37
Features nuevas creadas: 24
Total features: 61

📋 Lista de nuevas features:
   1. avg_grade_1st_sem
   2. avg_grade_2nd_sem
   3. avg_grade_overall
   4. approval_rate_1st_sem
   5. approval_rate_2nd_sem
   6. approval_rate_overall
   7. total_approved
   8. total_enrolled
   9. grade_improvement
  10. approval_improvement
  11. eval_to_approved_1st
  12. eval_to_approved_2nd
  13. at_risk_1st_sem
  14. very_low_performance
  15. critical_approval
  16. performance_declining
  17. economic_risk
  18. risk_score
  19. socioeconomic_index
  20. age_deviation
  21. is_mature_student
  22. is_very_young
  23. parents_education_avg
  24. economic_stress


In [7]:
# Correlación de nuevas features con target (ANOVA para categórico)
from scipy.stats import f_oneway

print("\n🎯 Importancia de nuevas features (ANOVA F-Statistic):")
print("="*60)

new_numeric_features = [col for col in new_cols 
                        if df_engineered[col].dtype in ['int64', 'float64']]

anova_results = []
for col in new_numeric_features:
    groups = [df_engineered[df_engineered['Target'] == cat][col].dropna() 
              for cat in df_engineered['Target'].unique()]
    
    if all(len(g) > 0 for g in groups):
        f_stat, p_value = f_oneway(*groups)
        anova_results.append({
            'Feature': col,
            'F_Statistic': f_stat,
            'P_Value': p_value
        })

anova_df = pd.DataFrame(anova_results).sort_values('F_Statistic', ascending=False)
print(anova_df.head(10).to_string(index=False))

print(f"\n✅ Features nuevas significativas (p<0.05): {(anova_df['P_Value'] < 0.05).sum()}/{len(anova_df)}")


🎯 Importancia de nuevas features (ANOVA F-Statistic):
              Feature  F_Statistic       P_Value
approval_rate_2nd_sem  2137.157448  0.000000e+00
approval_rate_overall  1986.525915  0.000000e+00
approval_rate_1st_sem  1444.747454  0.000000e+00
           risk_score  1340.975000  0.000000e+00
    critical_approval  1302.237598  0.000000e+00
       total_approved  1182.700830  0.000000e+00
    avg_grade_2nd_sem  1134.109544  0.000000e+00
    avg_grade_overall  1025.884406  0.000000e+00
 very_low_performance   967.864315  0.000000e+00
    avg_grade_1st_sem   713.517328 2.803052e-269

✅ Features nuevas significativas (p<0.05): 23/24


## 🔧 7. CONSTRUCCIÓN DEL PIPELINE DE PREPROCESSING

Pipeline completo con transformadores personalizados.

In [8]:
# Transformer personalizado para Feature Engineering
class FeatureEngineer(BaseEstimator, TransformerMixin):
    """Aplica todas las transformaciones de feature engineering"""
    
    def __init__(self):
        pass
    
    def fit(self, X, y=None):
        # Calcular mediana de edad en training set
        self.median_age_ = X['Age at enrollment'].median()
        return self
    
    def transform(self, X):
        X = X.copy()
        
        # Académicas
        X['avg_grade_overall'] = (X['Curricular units 1st sem (grade)'] + 
                                   X['Curricular units 2nd sem (grade)']) / 2
        
        X['approval_rate_overall'] = (
            np.where(X['Curricular units 1st sem (enrolled)'] > 0,
                     X['Curricular units 1st sem (approved)'] / X['Curricular units 1st sem (enrolled)'], 0) +
            np.where(X['Curricular units 2nd sem (enrolled)'] > 0,
                     X['Curricular units 2nd sem (approved)'] / X['Curricular units 2nd sem (enrolled)'], 0)
        ) / 2
        
        X['total_approved'] = (X['Curricular units 1st sem (approved)'] + 
                                X['Curricular units 2nd sem (approved)'])
        
        X['grade_improvement'] = (X['Curricular units 2nd sem (grade)'] - 
                                   X['Curricular units 1st sem (grade)'])
        
        # Riesgo
        X['at_risk_1st_sem'] = (
            (X['Curricular units 1st sem (approved)'] < 4) & 
            (X['Curricular units 1st sem (grade)'] < 11)
        ).astype(int)
        
        X['risk_score'] = (
            X['at_risk_1st_sem'] * 3 +
            (X['avg_grade_overall'] < 10).astype(int) * 2 +
            (X['approval_rate_overall'] < 0.5).astype(int) * 2
        )
        
        # Económicas
        X['socioeconomic_index'] = (
            X['Scholarship holder'] * 2 +
            X['Tuition fees up to date'] * 1 -
            X['Debtor'] * 2
        )
        
        X['age_deviation'] = X['Age at enrollment'] - self.median_age_
        
        return X

print("✅ FeatureEngineer transformer creado")

✅ FeatureEngineer transformer creado


In [9]:
# Definir columnas por tipo
target_col = 'Target'

# Columnas numéricas originales (continuas)
numeric_features = [
    'Age at enrollment',
    'Previous qualification (grade)',
    'Admission grade',
    'Curricular units 1st sem (credited)',
    'Curricular units 1st sem (enrolled)',
    'Curricular units 1st sem (evaluations)',
    'Curricular units 1st sem (approved)',
    'Curricular units 1st sem (grade)',
    'Curricular units 2nd sem (credited)',
    'Curricular units 2nd sem (enrolled)',
    'Curricular units 2nd sem (evaluations)',
    'Curricular units 2nd sem (approved)',
    'Curricular units 2nd sem (grade)',
    'Unemployment rate',
    'Inflation rate',
    'GDP'
]

# Features creadas por FeatureEngineer
engineered_features = [
    'avg_grade_overall',
    'approval_rate_overall',
    'total_approved',
    'grade_improvement',
    'at_risk_1st_sem',
    'risk_score',
    'socioeconomic_index',
    'age_deviation'
]

# Columnas categóricas
categorical_features = [
    'Marital status',
    'Application mode',
    'Application order',
    'Course',
    'Daytime/evening attendance\t',
    'Previous qualification',
    'Nacionality',
    "Mother's qualification",
    "Father's qualification",
    'Gender',
    'Scholarship holder',
    'Debtor',
    'Tuition fees up to date',
    'Displaced'
]

# Filtrar solo las que existen
numeric_features = [f for f in numeric_features if f in df.columns]
categorical_features = [f for f in categorical_features if f in df.columns]

print(f"Features numéricas originales: {len(numeric_features)}")
print(f"Features engineered: {len(engineered_features)}")
print(f"Features categóricas: {len(categorical_features)}")

Features numéricas originales: 16
Features engineered: 8
Features categóricas: 14


In [10]:
# Crear pipeline completo
from sklearn.preprocessing import RobustScaler

# Preprocesador para numéricas (original + engineered)
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# Preprocesador para categóricas
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num_original', numeric_transformer, numeric_features),
        ('num_engineered', numeric_transformer, engineered_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='drop'
)

# Pipeline completo
preprocessing_pipeline = Pipeline(steps=[
    ('feature_engineer', FeatureEngineer()),
    ('preprocessor', preprocessor)
])

print("✅ Pipeline de preprocessing creado:")
print(preprocessing_pipeline)

✅ Pipeline de preprocessing creado:
Pipeline(steps=[('feature_engineer', FeatureEngineer()),
                ('preprocessor',
                 ColumnTransformer(transformers=[('num_original',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['Age at enrollment',
                                                   'Previous qualification '
                                                   '(grade)',
                                                   'Admission grade',
                                                   'Curricular units 1st sem '
                                                   '(credited)',
                                                   'Curricular units 1st sem '
                                                   '(enrolled)',
                                                   'Curricular units 1st s

## 🧪 8. TESTING DEL PIPELINE

In [11]:
# Separar features y target
X = df_engineered.drop(columns=[target_col])
y = df_engineered[target_col]

# Encode target
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print(f"Target encoding:")
for i, label in enumerate(label_encoder.classes_):
    print(f"  {label} → {i}")

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"\n📊 Split:")
print(f"  Train: {X_train.shape[0]} samples")
print(f"  Test: {X_test.shape[0]} samples")

Target encoding:
  Dropout → 0
  Enrolled → 1
  Graduate → 2

📊 Split:
  Train: 3539 samples
  Test: 885 samples


In [12]:
# Fit pipeline en train
print("🔧 Fitting preprocessing pipeline...")
X_train_transformed = preprocessing_pipeline.fit_transform(X_train)
X_test_transformed = preprocessing_pipeline.transform(X_test)

print(f"\n✅ Pipeline fitted:")
print(f"  Train shape: {X_train_transformed.shape}")
print(f"  Test shape: {X_test_transformed.shape}")
print(f"\nTotal features después del pipeline: {X_train_transformed.shape[1]}")

🔧 Fitting preprocessing pipeline...

✅ Pipeline fitted:
  Train shape: (3539, 181)
  Test shape: (885, 181)

Total features después del pipeline: 181


## 💾 9. GUARDAR PIPELINE Y DATOS PROCESADOS

In [13]:
import joblib
import os

# Crear directorios
os.makedirs('../models', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

# Guardar pipeline
joblib.dump(preprocessing_pipeline, '../models/preprocessing_pipeline.pkl')
joblib.dump(label_encoder, '../models/label_encoder.pkl')

print("✅ Pipeline guardado: models/preprocessing_pipeline.pkl")
print("✅ Label encoder guardado: models/label_encoder.pkl")

# Guardar datos procesados
np.save('../data/processed/X_train.npy', X_train_transformed)
np.save('../data/processed/X_test.npy', X_test_transformed)
np.save('../data/processed/y_train.npy', y_train)
np.save('../data/processed/y_test.npy', y_test)

print("\n✅ Datos procesados guardados:")
print("  • data/processed/X_train.npy")
print("  • data/processed/X_test.npy")
print("  • data/processed/y_train.npy")
print("  • data/processed/y_test.npy")

✅ Pipeline guardado: models/preprocessing_pipeline.pkl
✅ Label encoder guardado: models/label_encoder.pkl

✅ Datos procesados guardados:
  • data/processed/X_train.npy
  • data/processed/X_test.npy
  • data/processed/y_train.npy
  • data/processed/y_test.npy


In [15]:
# Verificar dimensiones finales
import numpy as np

X_train = np.load('../data/processed/X_train.npy')
X_test = np.load('../data/processed/X_test.npy')

print(f"✅ Train: {X_train.shape}")
print(f"✅ Test: {X_test.shape}")
print(f"\nTotal features después del pipeline: {X_train.shape[1]}")

✅ Train: (3539, 181)
✅ Test: (885, 181)

Total features después del pipeline: 181


## 📋 10. DOCUMENTACIÓN FINAL

In [14]:
# Crear reporte de feature engineering
feature_engineering_report = {
    'original_features': original_features,
    'engineered_features': new_features,
    'total_features_after_pipeline': int(X_train_transformed.shape[1]),
    'train_samples': int(X_train.shape[0]),
    'test_samples': int(X_test.shape[0]),
    'target_encoding': {label: int(i) for i, label in enumerate(label_encoder.classes_)},
    'top_engineered_features': anova_df.head(5)['Feature'].tolist(),
    'preprocessing_steps': [
        '1. Feature Engineering (custom transformer)',
        '2. StandardScaler for numeric features',
        '3. OneHotEncoder for categorical features'
    ],
    'next_step': 'Model Training with CatBoost/LightGBM'
}

with open('../../reports/feature_engineering_report.json', 'w') as f:
    json.dump(feature_engineering_report, f, indent=2)

print("="*60)
print("FEATURE ENGINEERING COMPLETADO")
print("="*60)
print(f"✅ {new_features} nuevas features creadas")
print(f"✅ Pipeline guardado y listo para producción")
print(f"✅ Datos procesados guardados")
print(f"✅ Reporte guardado: reports/feature_engineering_report.json")
print(f"\n🚀 Siguiente paso: Día 4 - Model Training")

FEATURE ENGINEERING COMPLETADO
✅ 24 nuevas features creadas
✅ Pipeline guardado y listo para producción
✅ Datos procesados guardados
✅ Reporte guardado: reports/feature_engineering_report.json

🚀 Siguiente paso: Día 4 - Model Training
